In [9]:
from typing import Literal
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pathlib import Path
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain.tools import tool
from langchain.chat_models import init_chat_model
import json


file_path = Path.cwd().parent / 'data' / 'cinderela.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    cind = f.read()

In [10]:


# ====== DEFINIÇÕES DE PROPP ======
PROPP_ROLES_DEFINITIONS = """
1. AGRESSOR (VILÃO): Causa o dano inicial ou a "falta" que inicia a história; entra em combate direto com o herói.
2. DOADOR (PROVEDOR): Testa o herói (interroga ou propõe desafio) e lhe fornece um objeto ou agente mágico.
3. AUXILIAR (AJUDANTE): Desloca o herói no espaço (voo, viagem rápida), resgata-o de perseguição, resolve tarefas difíceis ou transfigura o herói.
4. PRINCESA (E PAI): O objetivo da busca. O Pai atribui tarefas difíceis e entrega a Princesa. A Princesa é quem se casa com o herói.
5. MANDANTE: Aquele que percebe a falta e despacha o herói para a aventura (faz o chamado).
6. HERÓI: Aquele que parte na busca (buscador) ou sofre a agressão inicial (vítima) e reata a ela; casa-se no final.
7. FALSO HERÓI: Reivindica falsamente os feitos do herói ou tenta se casar com a princesa através de engano; geralmente é desmascarado.
"""

# ====== ESTADO DO GRAFO ======
class GraphState(TypedDict):
    conto: str
    personagens: list[str]
    classificacoes: dict
    justificativas: dict
    resultado_final: str

# ====== INICIALIZAR MODELO ======
def obter_modelo():
    """Inicializa o modelo de forma limpa."""
    return init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        temperature=0
    )

# ====== NÓS DO GRAFO ======
def node_listar_personagens(state: GraphState) -> GraphState:
    """Nó 1: Lista todos os personagens do conto."""
    print("\n📖 Etapa 1: Listando personagens...")
    
    model = obter_modelo()
    
    prompt = f"""
Analise este conto e liste TODOS os personagens mencionados (nomes próprios ou descrições como "O Rei", "A Bruxa", etc).

Conto:
{state["conto"]}

Responda com um JSON simples: {{"personagens": ["nome1", "nome2", ...]}}
Responda APENAS com o JSON, sem nenhum texto adicional.
"""
    
    response = model.invoke(prompt)
    
    try:
        result = json.loads(response.content)
        state["personagens"] = result.get("personagens", [])
    except json.JSONDecodeError:
        print("⚠️  Falha ao parsear JSON, tentando extração simples...")
        state["personagens"] = ["Personagem não identificado"]
    
    print(f"✓ Personagens encontrados: {state['personagens']}")
    return state

def node_classificar_propp(state: GraphState) -> GraphState:
    """Nó 2: Classifica personagens segundo Propp."""
    print("\n🎭 Etapa 2: Classificando personagens segundo Propp...")
    
    model = obter_modelo()
    
    personagens_str = ", ".join(state["personagens"])
    
    prompt = f"""
Você é um especialista em Narratologia.
Sua tarefa é analisar contos e classificar os personagens segundo as 7 Esferas de Ação de Vladimir Propp.

AS REGRAS DE PROPP:
{PROPP_ROLES_DEFINITIONS}

Nota: Um personagem pode ter múltiplos papéis.
Nota: Nem todos os papéis precisam estar presentes no conto.

Conto a analisar:
{state["conto"]}

Personagens para classificar: {personagens_str}

Responda com um JSON exatamente neste formato:
{{
    "classificacoes": {{
        "nome_personagem": ["PAPEL1", "PAPEL2"],
        ...
    }},
    "justificativas": {{
        "nome_personagem": {{
            "PAPEL1": "explicação breve",
            "PAPEL2": "explicação breve"
        }},
        ...
    }}
}}

Responda APENAS com o JSON, sem nenhum texto adicional.
"""
    
    response = model.invoke(prompt)
    
    try:
        result = json.loads(response.content)
        state["classificacoes"] = result.get("classificacoes", {})
        state["justificativas"] = result.get("justificativas", {})
    except json.JSONDecodeError as e:
        print(f"⚠️  Erro ao parsear JSON: {e}")
        state["classificacoes"] = {p: [] for p in state["personagens"]}
        state["justificativas"] = {p: {} for p in state["personagens"]}
    
    print("✓ Classificações concluídas")
    return state

def node_formatar_resultado(state: GraphState) -> GraphState:
    """Nó 3: Formata o resultado final."""
    print("\n📊 Etapa 3: Formatando resultado...")
    
    resultado = "=" * 70 + "\n"
    resultado += "ANÁLISE NARRATOLÓGICA - ESFERAS DE AÇÃO DE VLADIMIR PROPP\n"
    resultado += "=" * 70 + "\n\n"
    
    for personagem in state["personagens"]:
        resultado += f"👤 {personagem.upper()}\n"
        resultado += "─" * 70 + "\n"
        
        papéis = state["classificacoes"].get(personagem, [])
        if papéis:
            resultado += "   Papéis atribuídos:\n"
            for papel in papéis:
                justificativa = state["justificativas"].get(personagem, {}).get(papel, "")
                resultado += f"      • {papel}\n"
                if justificativa:
                    resultado += f"        └─ {justificativa}\n"
        else:
            resultado += "   ℹ️  Nenhum papel narrativo principal atribuído\n"
        
        resultado += "\n"
    
    state["resultado_final"] = resultado
    print("✓ Resultado formatado")
    
    return state

# ====== CONSTRUIR O GRAFO ======
def construir_grafo():
    """Constrói o grafo de análise narratológica."""
    
    graph = StateGraph(GraphState)
    
    # Adicionar nós
    graph.add_node("listar_personagens", node_listar_personagens)
    graph.add_node("classificar_propp", node_classificar_propp)
    graph.add_node("formatar_resultado", node_formatar_resultado)
    
    # Adicionar arestas (fluxo)
    graph.add_edge(START, "listar_personagens")
    graph.add_edge("listar_personagens", "classificar_propp")
    graph.add_edge("classificar_propp", "formatar_resultado")
    graph.add_edge("formatar_resultado", END)
    
    return graph.compile()

In [11]:
# ====== EXECUTAR ======
# Exemplo: Chapeuzinho Vermelho
conto_exemplo = """
Era uma vez uma menina chamada Chapeuzinho Vermelho que vivia com sua mãe 
numa cabana perto da floresta. Um dia, a mãe pediu que ela levasse uma cesta 
com comida para a avó que estava doente. No caminho, encontrou um lobo 
feroz que a enganou, fingindo ser inofensivo. O lobo correu para a casa 
da avó, a engoliu e se disfarçou na cama. Quando Chapeuzinho chegou, 
conversou com o "lobo-avó" mas logo desconfiou. Nesse momento, um caçador 
que passava pela floresta ouviu barulhos, invadiu a casa e matou o lobo, 
libertando a avó. Chapeuzinho aprendeu a nunca mais confiar em estranhos.
"""

print("\n🚀 Iniciando análise narratológica...\n")

# Construir e executar o grafo
grafo = construir_grafo()

resultado = grafo.invoke({
    "conto": conto_exemplo,
    "personagens": [],
    "classificacoes": {},
    "justificativas": {},
    "resultado_final": ""
})

# Exibir resultado formatado
print("\n" + resultado["resultado_final"])

# Exibir dados estruturados em JSON
print("\n📋 DADOS ESTRUTURADOS (JSON):")
print(json.dumps({
    "personagens": resultado["personagens"],
    "classificacoes": resultado["classificacoes"],
    "justificativas": resultado["justificativas"]
}, ensure_ascii=False, indent=2))


🚀 Iniciando análise narratológica...


📖 Etapa 1: Listando personagens...
✓ Personagens encontrados: ['Chapeuzinho Vermelho', 'mãe', 'avó', 'lobo', 'caçador']

🎭 Etapa 2: Classificando personagens segundo Propp...
✓ Classificações concluídas

📊 Etapa 3: Formatando resultado...
✓ Resultado formatado

ANÁLISE NARRATOLÓGICA - ESFERAS DE AÇÃO DE VLADIMIR PROPP

👤 CHAPEUZINHO VERMELHO
──────────────────────────────────────────────────────────────────────
   Papéis atribuídos:
      • HERÓI
        └─ Ela parte na busca para levar comida à avó.
      • PRINCESA
        └─ Ela é o objetivo da busca, pois é a neta da avó.

👤 MÃE
──────────────────────────────────────────────────────────────────────
   Papéis atribuídos:
      • MANDANTE
        └─ Ela é quem percebe a necessidade de Chapeuzinho levar comida à avó.

👤 AVÓ
──────────────────────────────────────────────────────────────────────
   Papéis atribuídos:
      • PRINCESA
        └─ Ela é a figura que Chapeuzinho busca ajudar.

👤 LOBO
